# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(metadata.name + ": " + metadata.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate all available RecordSets and display the fields (columns) within each RecordSet, referencing their `@id` attributes.

In [ ]:
# Explore available RecordSets in the dataset
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', '-')}")
        if rs.fields:
            print("  Fields / Columns:")
            for field in rs.fields:
                print(f"    - {field.name} (Field @id: {field.id}, DataType: {getattr(field, 'data_type', '-')})")
        print("\n")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Reference the record set and field `@id`s as shown above.

*For this dataset, our primary RecordSet is usually the protein matrix. Let us extract all RecordSets whose name contains 'protein', 'main', or similar.*

In [ ]:
# Collect the @id's of all RecordSets for extraction
record_set_ids = [rs.id for rs in getattr(metadata, 'record_sets', [])]
# For demonstration, extract all record sets (for small datasets), otherwise select as needed
dataframes = dict()

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  => {len(df)} records. Columns: {df.columns.tolist()}")
        # Only display head if small
        display(df.head())
    else:
        print(f"  => No records found in RecordSet {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

---
**Below, we select a RecordSet and numeric field for demonstration, using field `@id`. Please update these variables (`analyzed_record_set_id`, `numeric_field_id`, `group_field_id`) to match the chosen RecordSet and available fields in your dataset as printed above.**

In [ ]:
# === SELECT RECORDSET AND FIELDS FOR EDA === #
# Choose from the display above. Example RecordSet @id:
analyzed_record_set_id = record_set_ids[0] if record_set_ids else None

if analyzed_record_set_id and analyzed_record_set_id in dataframes:
    df = dataframes[analyzed_record_set_id]
    print(f"Analysis on RecordSet: {analyzed_record_set_id}")
    print(f"Available columns: {df.columns.tolist()}")
    # Choose a numeric field, for demonstration try MW or any %Abundance
    possible_numeric = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int]]
    numeric_field = possible_numeric[0] if possible_numeric else None
    print(f"Selected numeric field: {numeric_field}")
    # Select a group field if any
    group_cands = [c for c in df.columns if 'group' in c.lower() or 'condition' in c.lower() or 'sample' in c.lower() or 'gene' in c.lower()]
    group_field = group_cands[0] if group_cands else df.columns[0]
    print(f"Selected group field: {group_field}")
    
    # --- Example Filtering and Normalization ---
    if numeric_field:
        if df[numeric_field].dtype != np.number:
            # Try to convert to numeric
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean() # e.g. filter above mean
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records where {numeric_field} > {threshold:.2f} (mean)")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (mean 0, std 1):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Optional: Grouping
        if group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean by {group_field}:")
            print(grouped_df.head())
else:
    print("No RecordSet or DataFrame to analyze.")

## 5. Visualization
Visualize the data distribution of the numeric field and the results of the grouping.

*Update field names as required for your dataset's columns.*

In [ ]:
if analyzed_record_set_id and analyzed_record_set_id in dataframes:
    df = dataframes[analyzed_record_set_id]
    if 'filtered_df' in locals() and numeric_field:
        plt.figure(figsize=(8,4))
        plt.hist(df[numeric_field].dropna(), bins=30, color='steelblue', alpha=0.7)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()
        if group_field and group_field in df.columns:
            plt.figure(figsize=(10, 5))
            plot_df = df.groupby(group_field)[numeric_field].mean().sort_values()
            plot_df.plot(kind='bar', color='orange')
            plt.title(f"Mean {numeric_field} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(f"Mean {numeric_field}")
            plt.tight_layout()
            plt.show()

## 6. Conclusion
We have demonstrated how to load, explore, and process record sets from this FAIR² dataset using the `mlcroissant` Python library.

**Key findings:**
- The dataset provides mass spectrometry protein data across multiple fields, supporting flexible analysis.
- Fields and grouping attributes should be referenced by their Croissant schema `@id` for reliable and reproducible processing.
- Data can be filtered, normalized, and visualized directly with standard Python tools.

For deeper analysis, consult the Croissant schema for complete @id mappings and refer to dataset documentation for biological interpretation.